In [35]:
import pandas as pd
import sys
sys.path.append("./modules")
from modules.Cleaning import *
from modules.Functions import *

# Cleaning the data
### 1. Characters df
##### Removing columns that are not needed

In [36]:
dfCharacters = pd.read_csv("../data/raw/characters.csv")
dfCharacters = clean_characters(dfCharacters)
dfCharacters

,character_mal_id,name,image,favorites
0,280386,Envi Mel Champagne,https://cdn.myanimelist.net/images/characters/...,0
1,280354,Eleven,https://cdn.myanimelist.net/images/characters/...,0
2,280353,Stud,https://cdn.myanimelist.net/images/characters/...,0
3,280352,Judge,https://cdn.myanimelist.net/images/characters/...,0
4,280339,Eiji Kurokawa,https://cdn.myanimelist.net/img/sp/icon/apple-...,0
...,...,...,...,...
209958,282276,Farrah Van Dorothy,https://cdn.myanimelist.net/images/characters/...,0
209959,282277,Harris Mead,https://cdn.myanimelist.net/images/characters/...,0
209960,282278,Rob,https://cdn.myanimelist.net/images/characters/...,0
209961,282281,Grimm,https://cdn.myanimelist.net/images/characters/...,0


### 2. Nicknames df
##### Removed rows where nickname was empty or Nan, useless
##### Some Characters have multiple nicknames, so it's better to put them in the same row as a list

In [37]:
dfNicknames = pd.read_csv("../data/raw/character_nicknames.csv")
dfNicknames = clean_nicknames(dfNicknames)
dfNicknames

,character_mal_id,nickname
0,3,"[Running Rock, Black Dog]"
1,5,"[Ichi-nii, Shinigami Daiko (Substitute Soul Re..."
2,7,[Hime]
3,11,"[Ed, Fullmetal Alchemist, Hagane no shounen, C..."
4,12,"[Al, Armored Alchemist]"
...,...,...
30262,282048,[Great Zuma]
30263,282159,"[Ling Long, Silvermoon]"
30264,282227,[Mei's Mother]
30265,282254,[Cyrano]


In [38]:
dfCharacters = dfCharacters.merge(dfNicknames, on="character_mal_id", how = "left")
dfNicknames = None

In [39]:
dfCharacters['nickname'] = dfCharacters['nickname'].apply(lambda x: x if isinstance(x, list) else [])

### 3. AnimeWorks df

In [40]:
dfAnimeWorks = pd.read_csv("../data/raw/character_anime_works.csv")
dfAnimeWorks

,anime_mal_id,character_mal_id,character_name,role
0,2928,5781,Atoli,Main
1,2928,33,Haseo,Main
2,2928,32,Ovan,Main
3,2928,34,Shino,Main
4,2928,5785,Aina,Supporting
...,...,...,...,...
236811,31245,137157,"Shibasaki, Ken",Supporting
236812,36305,136064,"Hamanaka, Midori",Main
236813,36305,133916,"Narumi, Sena",Main
236814,36305,124942,"Hayasaka, Akari",Supporting


In [41]:

dfCharacters = dfCharacters.merge(dfAnimeWorks, on="character_mal_id", how="left")
dfAnimeWorks = None

dfCharacters = dfCharacters.drop(columns=["name"]) # Redundant column, keeping the one with the comma
dfCharacters

,character_mal_id,image,favorites,nickname,anime_mal_id,character_name,role
0,280386,https://cdn.myanimelist.net/images/characters/...,0,[],59846.0,"Champagne, Envi Mel",Supporting
1,280354,https://cdn.myanimelist.net/images/characters/...,0,[],60071.0,Eleven,Supporting
2,280353,https://cdn.myanimelist.net/images/characters/...,0,[],60071.0,Stud,Supporting
3,280352,https://cdn.myanimelist.net/images/characters/...,0,[],60071.0,Judge,Supporting
4,280339,https://cdn.myanimelist.net/img/sp/icon/apple-...,0,[],60531.0,"Kurokawa, Eiji",Supporting
...,...,...,...,...,...,...,...
303384,282276,https://cdn.myanimelist.net/images/characters/...,0,[],3258.0,"Van Dorothy, Farrah",Supporting
303385,282277,https://cdn.myanimelist.net/images/characters/...,0,[],3258.0,"Mead, Harris",Supporting
303386,282278,https://cdn.myanimelist.net/images/characters/...,0,[],7625.0,Rob,Supporting
303387,282281,https://cdn.myanimelist.net/images/characters/...,0,[],7625.0,Grimm,Supporting


Putting all anime where a character participated in a list.


In [42]:
dfTemp = (
    dfCharacters
    .groupby(["character_mal_id", "role"])
    .agg({
        "favorites": "max",
        "anime_mal_id": lambda x: [int(v) for v in x.unique()]
    })
    .reset_index()
)
dfTemp

,character_mal_id,role,favorites,anime_mal_id
0,1,Main,48344,"[1, 5, 4037]"
1,2,Main,9535,"[1, 5, 4037]"
2,3,Main,2249,"[1, 5, 4037]"
3,4,Main,2391,[17205]
4,4,Supporting,2391,"[1, 5, 4037]"
...,...,...,...,...
150575,282276,Supporting,0,[3258]
150576,282277,Supporting,0,[3258]
150577,282278,Supporting,0,[7625]
150578,282281,Supporting,0,[7625]


Taking the index for the best favorites

In [43]:
idx = (
    dfCharacters
    .groupby("character_mal_id")["favorites"]
    .idxmax()
)
idx

character_mal_id
1         300256
2         300253
3         300250
4         300246
5         300229
           ...  
282276    303384
282277    303385
282278    303386
282281    303387
282284    303388
Name: favorites, Length: 209506, dtype: int64

Tracking the best rows thanks to the idx and merging to get the other columns back

In [44]:
dfCharacters = dfCharacters.loc[idx][["character_mal_id", "nickname", "character_name"]]
dfTemp = dfTemp.merge(dfCharacters, on="character_mal_id", how="left")
dfTemp

,character_mal_id,role,favorites,anime_mal_id,nickname,character_name
0,1,Main,48344,"[1, 5, 4037]",[],"Spiegel, Spike"
1,2,Main,9535,"[1, 5, 4037]",[],"Valentine, Faye"
2,3,Main,2249,"[1, 5, 4037]","[Running Rock, Black Dog]","Black, Jet"
3,4,Main,2391,[17205],[],Ein
4,4,Supporting,2391,"[1, 5, 4037]",[],Ein
...,...,...,...,...,...,...
150575,282276,Supporting,0,[3258],[],"Van Dorothy, Farrah"
150576,282277,Supporting,0,[3258],[],"Mead, Harris"
150577,282278,Supporting,0,[7625],[],Rob
150578,282281,Supporting,0,[7625],[],Grimm


In [45]:
dfCharacters = None
save_csv(dfTemp, "../data/cleaned/anime_characters.csv")
dfTemp = None